# 1.5 Online Softmax 数值实验

1.4 节给了公式，本节给证据。三个实验层层递进：

1. **Softmax 一维版**：朴素 / 安全 / 在线三种实现，逐位对拍；
2. **完整 FlashAttention**：把 1.4 节伪代码翻译成 numpy（分块 + Online Softmax + 因果掩码），与标准 Attention 对拍；
3. **压力测试**：大分数（溢出边缘）、极端分块大小，观察数值行为。

所有代码只依赖 numpy，鼓励动手改参数。

## 实验一：三种 Softmax 的对拍

- `softmax_naive`：直接 exp——遇到大分数会溢出；
- `softmax_safe`：减行最大值——稳定，但需要**两遍扫描**（先 max 后 sum）；
- `softmax_online`：数据分成 B 块逐块到来，只维护 (m, l) 两个状态——**一遍扫描**，这正是 Kernel 里的工作方式。

In [ ]:
import numpy as np
np.random.seed(42)

def softmax_naive(x):
    e = np.exp(x)
    return e / e.sum()

def softmax_safe(x):
    e = np.exp(x - x.max())          # 两遍扫描：第一遍 max，第二遍 sum
    return e / e.sum()

def softmax_online(x, B):
    """数据分块到来，只用状态 (m, l) 递推，返回归一化结果。"""
    m, l = -np.inf, 0.0
    out = np.zeros_like(x)
    for i in range(0, len(x), B):    # 逐块处理
        blk = x[i:i+B]
        m_new = max(m, blk.max())            # ① 更新最大值
        alpha = np.exp(m - m_new)            #    修正因子（m=-inf 时首块 alpha=0，正好）
        p = np.exp(blk - m_new)              # ② 新块概率
        l = alpha * l + p.sum()              # ③ 重标定旧状态并累加
        out[i:i+B] = p
        m = m_new
        out[:i] *= alpha                     #    旧结果也按 alpha 缩小
    return out / l                          # 收尾：一次归一化

x = np.random.randn(16) * 3
B = 4
print('safe  与 naive 一致（小分数场景）:', np.allclose(softmax_safe(x), softmax_naive(x)))
print('online 与 safe   一致          :', np.allclose(softmax_online(x, B), softmax_safe(x)))
print('\nonline 结果:', np.round(softmax_online(x, B), 4))

safe  与 naive 一致（小分数场景）: True
online 与 safe   一致          : True

online 结果: [0.0184 0.0027 0.0289 0.3989 0.002  0.002  0.4722 0.0413 0.001  0.0211
 0.001  0.001  0.0085 0.     0.     0.0008]


三个实现结果一致。再单独看“递推过程”如何演进——注意 `alpha` 修正旧结果的时刻：

In [ ]:
def softmax_online_trace(x, B):
    m, l = -np.inf, 0.0
    for i in range(0, len(x), B):
        blk = x[i:i+B]
        m_new = max(m, blk.max())
        alpha = np.exp(m - m_new) if m > -np.inf else 0.0
        p = np.exp(blk - m_new)
        l = alpha * l + p.sum()
        print(f'块 {i//B}: 块内最大={blk.max():+7.2f}  新基准 m={m_new:+7.2f}'
              f'  alpha={alpha:.4f}  l={l:8.4f}')
        m = m_new

x_demo = np.array([1.0, 2.0, 3.0, 1.5,   6.0, 0.5, 2.0, 1.0,   0.0, 0.5, 4.0, 0.2,   0.1, 0.3, 0.4, 0.0])
softmax_online_trace(x_demo, 4)

块 0: 块内最大=  +3.00  新基准 m=  +3.00  alpha=0.0000  l=  1.7263
块 1: 块内最大=  +6.00  新基准 m=  +6.00  alpha=0.0498  l=  1.1151
块 2: 块内最大=  +4.00  新基准 m=  +6.00  alpha=1.0000  l=  1.2600
块 3: 块内最大=  +0.40  新基准 m=  +6.00  alpha=1.0000  l=  1.2723


观察：当第 2 块出现更大的最大值（6.0）时，`alpha` 变得远小于 1——前两块的贡献被整体缩小；此后基准不再被超越，`alpha = 1.0000`，退化为纯累加。**这就是“基准换了，旧账折算”的可视化。**

> 试试修改 `x_demo` 中各块的数值，看看什么情况下 `alpha` 频繁小于 1（提示：最大值不断被刷新时）。

## 实验二：完整的分块 FlashAttention

把 1.4 节的伪代码逐行翻译过来。注意与真实 Kernel 的对应关系已写在注释里。

In [ ]:
def attention_reference(Q, K, V, causal=False):
    """标准 Attention：整行 Softmax，两遍扫描（对照组）。"""
    n, d = Q.shape
    S = Q @ K.T / np.sqrt(d)
    if causal:
        S = S + np.triu(np.full((n, n), -np.inf), k=1)   # 上三角 -inf
    P = np.exp(S - np.where(np.isfinite(S), S, 0).max(axis=1, keepdims=True))
    P = P / P.sum(axis=1, keepdims=True)
    return P @ V

def flash_attention(Q, K, V, M, B, causal=False):
    """FlashAttention：外层 Q 块(每块 M 行) + 内层 KV 块(每块 B 行) + Online Softmax。
    数学上与 attention_reference 精确等价（M、B 无须整除 n）。"""
    n, d = Q.shape
    O = np.zeros_like(Q)
    for i0 in range(0, n, M):                       # ── 外层：Q 块（驻留片上）
        Qi = Q[i0:i0+M]                             #    (Mi, d)
        Mi = Qi.shape[0]
        m = np.full(Mi, -np.inf)                    # 状态①：行最大值
        l = np.zeros(Mi)                            # 状态②：指数和
        acc = np.zeros((Mi, d))                     # 状态③：未归一化输出

        j_max = min(i0 + Mi, n) if causal else n    # 因果掩码：块级裁剪！
        for j0 in range(0, j_max, B):               # ── 内层：KV 块（流过片上）
            Kj, Vj = K[j0:j0+B], V[j0:j0+B]         #    (Bj, d)
            Bj = Kj.shape[0]
            S = Qi @ Kj.T / np.sqrt(d)              # GEMM ①  (Mi, Bj)

            if causal and j0 + Bj > i0:             # 仅对角块需要元素级掩码
                rows = np.arange(i0, i0+Mi)[:, None]
                cols = np.arange(j0, j0+Bj)[None, :]
                S = np.where(cols > rows, -np.inf, S)

            blk_max = np.max(S, axis=1)             # -inf 行的 max 仍是 -inf，安全
            m_new = np.maximum(m, blk_max)          # 三步递推①
            alpha = np.exp(m - m_new)               # m=-inf → alpha=0（首块）
            P = np.exp(S - m_new[:, None])          # 三步递推②  (Mi, Bj)
            alpha = np.where(np.isnan(alpha), 0.0, alpha)   # inf-inf 的 NaN 兜底
            P = np.where(np.isneginf(S), 0.0, P)            # 掩码位 exp(-inf)=0

            l = alpha * l + P.sum(axis=1)           # 三步递推③
            acc = alpha[:, None] * acc + P @ Vj     # GEMM ②  (Mi, d)
            m = m_new

        O[i0:i0+Mi] = acc / l[:, None]              # 收尾：写回该 Q 块
    return O

In [ ]:
# 对拍：无掩码 / 有掩码，多种分块大小
n, d = 64, 16
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

for causal in [False, True]:
    ref = attention_reference(Q, K, V, causal)
    print(f'--- causal = {causal} ---')
    for M, B in [(16, 16), (8, 8), (7, 5), (1, 1), (64, 1)]:
        out = flash_attention(Q, K, V, M, B, causal)
        err = np.abs(out - ref).max()
        print(f'  M={M:3d}, B={B:3d}  →  最大误差 = {err:.2e}   一致: {err < 1e-10}')

--- causal = False ---
  M= 16, B= 16  →  最大误差 = 4.44e-16   一致: True
  M=  8, B=  8  →  最大误差 = 5.55e-16   一致: True
  M=  7, B=  5  →  最大误差 = 4.44e-16   一致: True
  M=  1, B=  1  →  最大误差 = 1.11e-15   一致: True
  M= 64, B=  1  →  最大误差 = 6.66e-16   一致: True
--- causal = True ---
  M= 16, B= 16  →  最大误差 = 5.55e-16   一致: True
  M=  8, B=  8  →  最大误差 = 4.44e-16   一致: True
  M=  7, B=  5  →  最大误差 = 4.44e-16   一致: True
  M=  1, B=  1  →  最大误差 = 1.11e-15   一致: True
  M= 64, B=  1  →  最大误差 = 7.77e-16   一致: True


无论分块多大（哪怕 `M=1, B=1` 退化成逐元素），结果都与标准实现**误差在 1e-10 量级**（纯浮点舍入差异）——FlashAttention 是精确算法的直接证据。

## 实验三：压力测试

### 3.1 大分数（溢出边缘）

把得分放大到 fp32 的 exp 溢出边缘，看三种实现谁还活着。

In [ ]:
n, d = 64, 16
scale = 12.0                       # 放大分数，逼近 exp 溢出区
Q = np.random.randn(n, d) * scale
K = np.random.randn(n, d) * scale
V = np.random.randn(n, d)

S = Q @ K.T / np.sqrt(d)
print(f'得分范围：[{S.min():.1f}, {S.max():.1f}]（fp32 的 exp 上溢点约 88）')

ref = attention_reference(Q, K, V)          # 内部已用 safe softmax
out = flash_attention(Q, K, V, M=16, B=16)
print('FlashAttention 与参考实现最大误差:', np.abs(out - ref).max())
print('结果是否有 nan/inf :', np.isnan(out).any() or np.isinf(out).any())

得分范围：[-747.6, 538.1]（fp32 的 exp 上溢点约 88）
FlashAttention 与参考实现最大误差: 6.661338147750939e-16
结果是否有 nan/inf : False


### 3.2 分块大小 vs 数值误差

分块大小理论上不影响正确性，但**极小分块会多经历很多次“重标定”，舍入误差路径不同**。观察误差量级（仍然远小于任何实际精度要求）：

In [ ]:
n, d = 256, 32
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)
ref = attention_reference(Q, K, V, causal=True)

for M in [256, 64, 16, 4, 1]:
    errs = [np.abs(flash_attention(Q, K, V, M, B, causal=True) - ref).max()
            for B in [256, 64, 16, 4, 1]]
    print(f'M={M:4d}: ' + '  '.join(f'{e:.1e}' for e in errs))
print('        （列对应 B=256/64/16/4/1；所有误差均在浮点舍入量级）')

M= 256: 6.1e-16  5.6e-16  8.9e-16  7.2e-16  8.9e-16
M=  64: 6.1e-16  5.6e-16  8.9e-16  7.2e-16  8.9e-16
M=  16: 6.1e-16  8.9e-16  8.9e-16  7.2e-16  8.9e-16
M=   4: 8.9e-16  8.9e-16  8.9e-16  7.2e-16  8.9e-16
M=   1: 7.8e-16  7.8e-16  8.9e-16  1.1e-15  8.9e-16
        （列对应 B=256/64/16/4/1；所有误差均在浮点舍入量级）


### 3.3 因果掩码到底省了多少计算？

数一数内层循环实际执行的块数，与 1.4 节的图对上：

In [ ]:
def count_blocks(n, M, B, causal):
    total = 0
    for i0 in range(0, n, M):
        j_max = i0 + M if causal else n
        total += len(range(0, j_max, B))
    return total

n = 4096
for M, B in [(128, 128), (256, 64)]:
    full = count_blocks(n, M, B, causal=False)
    causal_cnt = count_blocks(n, M, B, causal=True)
    print(f'M={M}, B={B}: 无掩码 {full} 块 → 因果掩码 {causal_cnt} 块，'
          f'节省 {(1 - causal_cnt/full)*100:.0f}% 的块计算')

M=128, B=128: 无掩码 1024 块 → 因果掩码 528 块，节省 48% 的块计算
M=256, B=64: 无掩码 1024 块 → 因果掩码 544 块，节省 47% 的块计算


省约一半，且 n 越大越接近理论值 50%（对角块占比趋于 0）——与 1.4 节因果掩码图完全对应。

## 本节小结

1. Online Softmax 与标准 Softmax **精确一致**（实验一、二）；
2. 分块大小 `M/B` 只影响性能与舍入路径，**不影响数学正确性**（实验 3.2）——这也是 Tiling 参数可以按硬件规格自由选取的前提；
3. Safe Softmax 机制让算法在**大分数下依然稳定**（实验 3.1）；
4. 因果掩码块级裁剪省约 50% 计算（实验 3.3）。

至此，第 1 章的因果链闭环：**Attention 是什么（1.2）→ 为什么慢（1.3）→ FlashAttention 怎么精确地变快（1.4）→ 亲手验证（1.5）**。



## 章节测验

1. 修改实验一中的 `B`（如 1、2、8、16），结论会变吗？为什么？
2. 实验 3.1 中若把 `scale` 加到 30，`attention_reference` 和 `flash_attention` 还能保持一致吗？各自内部依赖什么机制？
3. （找茬题）`flash_attention` 中若删掉 `alpha[:, None] *`（即 `acc` 不重标定），对拍会失败吗？构造一个必失败的场景（提示：让最大值只出现在后面的块）。
4. 实验 3.3 中把 `B` 调大（如 B=512），节省比例会如何变化？为什么？

> 答案见 `answer/01.05_answer.txt`。

恭喜完成第 1 章！返回 [章节目录](README.md)。